# Data Analysis 

## About this project
This notebook analyzes hotel booking data to understand the factors that 
influence whether a booking gets canceled. The goal is to explore the data, 
identify patterns and relationships, and prepare insights for the model-building 
stage.

## What this notebook covers:
1. Load the data and get a general overview
2. Check missing values and data types
3. Analyze the target variable distribution (canceled vs. not canceled)
4. Univariate analysis (distributions of key features)
5. Bivariate analysis (feature relationships with cancellation)
6. Correlation analysis
7. Outlier detection
8. Key insights / summary for feature engineering

## Dataset Column Descriptions

| Column | Description |
|--------|-------------|
| **hotel** | Type of hotel |
| **is_canceled** | Target variable — whether the booking was canceled (1) or not (0) |
| **lead_time** | Number of days between the booking date and the arrival (check-in) date |
| **arrival_date_year** | Year of arrival date |
| **arrival_date_month** | Month of arrival date |
| **arrival_date_week_number** | Week number of the year for the arrival date |
| **arrival_date_day_of_month** | Day of the month of the arrival date |
| **stays_in_weekend_nights** | Number of weekend nights (Saturday, Sunday) the guest stayed or booked |
| **stays_in_week_nights** | Number of week nights (Monday–Friday) the guest stayed or booked |
| **adults** | Number of adults included in the booking |
| **children** | Number of children included in the booking |
| **babies** | Number of babies included in the booking |
| **meal** | Type of meal plan booked (BB, HB, FB, SC, Undefined) |
| **country** | Country of origin of the guest |
| **market_segment** | Market segment designation (e.g., Online TA, Offline TA/TO, Direct, Corporate, Groups, Complementary, Aviation) |
| **distribution_channel** | Booking distribution channel (e.g., TA/TO, Direct, Corporate, GDS) |
| **is_repeated_guest** | Whether the guest is a repeat guest (1) or not (0) |
| **previous_cancellations** | Number of previous bookings that were canceled by the customer prior to this booking |
| **previous_bookings_not_canceled** | Number of previous bookings that were not canceled by the customer prior to this booking |
| **reserved_room_type** | Code of room type originally reserved |
| **assigned_room_type** | Code of room type actually assigned at check-in (may differ from reserved) |
| **booking_changes** | Number of changes/amendments made to the booking from the moment it was entered until check-in or cancellation |
| **deposit_type** | Type of deposit made — `No Deposit`, `Non Refund`, or `Refundable` |
| **agent** | ID of the travel agency that made the booking |
| **company** | ID of the company/entity that made the booking or is responsible for payment |
| **days_in_waiting_list** | Number of days the booking was on the waiting list before it was confirmed |
| **customer_type** | Type of booking — `Transient`, `Transient-Party`, `Contract`, or `Group` |
| **adr** | Average Daily Rate — average revenue earned per occupied room per day |
| **required_car_parking_spaces** | Number of car parking spaces required by the customer |
| **total_of_special_requests** | Number of special requests made by the customer (e.g., extra bed, high floor) |
| **reservation_status** | Last reservation status — `Canceled`, `Check-Out`, or `No-Show` |
| **reservation_status_date** | Date at which the last reservation status was set |
| **city** | City associated with the booking/hotel |

## Why Precision Matters More Than Recall for This Project

### Understanding the Trade-off

In this cancellation prediction task, two types of prediction errors are possible:

- **False Positive (FP):** The model predicts a booking will be **canceled**, but the guest actually **shows up**.
- **False Negative (FN):** The model predicts a booking will **not be canceled**, but the guest actually **cancels**.

**Precision** measures how many of the bookings flagged as "will cancel" actually do cancel. High precision means few false positives.

**Recall** measures how many of the actual cancellations the model successfully catches. High recall means few false negatives.

### Business Impact of Each Error Type

**If FP is high (low precision):**
The model incorrectly flags a booking as "will cancel." The hotel may act on this by reselling the room (overbooking). If the original guest then shows up, the hotel has no room available — leading to guest rejection, compensation costs, and reputational damage. This is often the most costly outcome in the hotel industry.

**If FN is high (low recall):**
The model fails to flag a booking that actually gets canceled. The hotel doesn't resell the room in advance, and it ends up empty — resulting in lost revenue, but no guest is turned away and no reputational harm occurs.

### Conclusion

Since falsely predicting a cancellation (FP) can lead to overbooking, guest rejection, and reputational damage — consequences that are typically more severe and costly than an empty room (FN) — **minimizing false positives, and therefore maximizing precision, is the priority for this project.**

This trade-off should guide:
- The choice of evaluation metric (prioritize Precision or F-beta with beta < 1)
- The classification threshold tuning (favor a higher threshold to reduce FPs)
- Model selection and comparison criteria going forward

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np


# Dataset Overview
def dataset_overview(df):
    """

    This function provides an overview of the dataset, including the number of rows and columns,
    data types of each column, and the number of missing values in each column.
    
    """
    print("Dataset Overview:")
    print(f"Info: {df.info()}")
    print(f"Number of rows: {df.shape[0]}")
    print(f"Number of columns: {df.shape[1]}")
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nSummary Statistics:")
    print(df.describe())

    print("\n Unique values:")
    for col in df.columns:
            unique_vals = df[col].unique()
            if len(unique_vals) <= 20:
                values_str = " ".join(str(v) for v in unique_vals)
                print(f"{col} => {values_str}")
            else:
                print(f"{col} => Too many unique values ({df[col].nunique()})")
    

# Missing Values Analysis
def missing_values_analysis(df):
    """

    This function analyzes the missing values in the dataset and provides insights into their distribution.
    
    """
    missing_values = df.isnull().sum()
    missing_values = missing_values[missing_values > 0]
    print("Missing Values Analysis:")
    print(missing_values)

    # Visualizing missing values
    px.bar(
        missing_values, x=missing_values.index, y=missing_values.values,
        labels={'x': 'Columns', 'y': 'Number of Missing Values'},
        title='Missing Values in Each Column',
        color=missing_values.index
    ).show()  

# target variable balance
def target_variable_balance(df, target_column):
    """

    This function visualizes the balance of the target variable in the dataset.

    """
    px.bar(
        df[target_column].value_counts(), x=df[target_column].value_counts().index,
        y=df[target_column].value_counts().values,
        labels={'x': target_column, 'y': 'Count'},
        title=f'Balance of Target Variable: {target_column}',
        color=df[target_column].value_counts().index
    ).show()

# distribution of adult, children, and babies
def numerical_features_distribution(df):
    """

    This function visualizes the distribution of children, adults, and babies in the dataset.

    """
    numerical_features = ['adults', 'children', 'babies']
    for feature in numerical_features:
        px.histogram(
            df, x=feature, nbins=30,
            labels={'x': feature, 'y': 'Count'},
            title=f'Distribution of {feature.capitalize()}',
        ).show()

# Meal and cancellation distribution
def meal_distribution(df):
    """
    This function visualizes the distribution of meal types and cancellation in the dataset.
    """

    def meal_definition(meal_code):
        meal_dict = {
            'BB': 'Bed & Breakfast (B&B)',
            'FB': 'Full Board (FB)',
            'HB': 'Half Board (HB)',
            'SC': 'Self Catering (SC)',
            'Undefined': 'Undefined'
        }
        return meal_dict.get(meal_code, 'Unknown')

    meal_cancellation_counts = df.groupby(['meal', 'is_canceled']).size().reset_index(name='count')

    meal_cancellation_counts['meal_name'] = meal_cancellation_counts['meal'].apply(meal_definition)
    meal_cancellation_counts['is_canceled'] = meal_cancellation_counts['is_canceled'].map({0: 'Not Canceled', 1: 'Canceled'})

    px.bar(
        meal_cancellation_counts,
        x='meal_name',
        y='count',
        color='is_canceled',
        barmode='group',
        labels={'meal_name': 'Meal Type', 'count': 'Count', 'is_canceled': 'Booking Status'},
        title='Meal Type Distribution by Cancellation Status'
    ).show()

# Customer type and cancellation distribution
def customer_type_cancellation_distribution(df):
    """

    This function visualizes the distribution of customer types and their cancellation status in the dataset.

    """
    customer_cancellation_counts = df.groupby(['customer_type', 'is_canceled']).size().reset_index(name='count')

    px.bar(
        customer_cancellation_counts,
        x='customer_type',
        y='count',
        color='is_canceled',
        labels={'customer_type': 'Customer Type', 'count': 'Count', 'is_canceled': 'Cancellation Status'},
        title='Customer Type and Cancellation Distribution',
        barmode='group'
    ).show()

# Market segment and cancellation distribution
def market_segment_cancellation_distribution(df):
    """

    This function visualizes the distribution of market segments and their cancellation status in the dataset.

    """
    market_cancellation_counts = df.groupby(['market_segment', 'is_canceled']).size().reset_index(name='count')

    px.bar(
        market_cancellation_counts,
        x='market_segment',
        y='count',
        color='is_canceled',
        labels={'market_segment': 'Market Segment', 'count': 'Count', 'is_canceled': 'Cancellation Status'},
        title='Market Segment and Cancellation Distribution',
        barmode='group'
    ).show()

# Distribution channel and cancellation distribution
def distribution_channel_cancellation_distribution(df):
    """

    This function visualizes the distribution of distribution_channel and their cancellation status in the dataset.

    """
    distribution_channel_counts = df.groupby(['distribution_channel', 'is_canceled']).size().reset_index(name='count')

    px.bar(
        distribution_channel_counts,
        x='distribution_channel',
        y='count',
        color='is_canceled',
        labels={'distribution_channel': 'Distribution channel', 'count': 'Count', 'is_canceled': 'Cancellation Status'},
        title='Distribution channel and Cancellation Distribution',
        barmode='group'
    ).show()

# Reserved room type, Assigned room type and Cancellation distribution
def room_type_cancellation(df):
    df["room_match"] = np.where(
    df["reserved_room_type"] == df["assigned_room_type"], "Same", "Changed")

    df["booking_status"] = df["is_canceled"].map(
    {0: "Check-in", 1: "Canceled"})

    df_grouped = (
    df.groupby(["room_match", "booking_status"])
    .size()
    .reset_index(name="count"))

    fig = px.bar(
        df_grouped,
        x="room_match",
        y="count",
        color="booking_status",
        barmode="group",
        title="Impact of room cancellation on the cancellation status",
        labels={
            "room_match": "Room status (Same or Changed)",
            "count": "Number of orders",
            "booking_status": "Order status",
        },
        color_discrete_map={
            "Check-in": "#2ecc71",
            "Canceled": "#e74c3c",
        },
    )

    fig.update_layout(xaxis_title="", yaxis_title="Number of orders", hovermode="x")

    fig.show()

# correlation 
def correlation(df):
    """
    Computes and visualizes the correlation matrix for numerical features.
    
    """
    numerical_features = df.select_dtypes(include=['number'])
    corr_matrix = numerical_features.corr()

    px.imshow(
        corr_matrix,
        text_auto='.2f',
        aspect='auto',
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        title='Correlation Matrix of Numerical Features'
    ).show()

    return corr_matrix


# Usage
if __name__ == "__main__":
    df=pd.read_csv("../data/raw/hotel_bookings_updated_2024.csv")
    dataset_overview(df)
    missing_values_analysis(df)
    target_variable_balance(df, 'is_canceled')
    numerical_features_distribution(df)
    meal_distribution(df)
    customer_type_cancellation_distribution(df)
    market_segment_cancellation_distribution(df)
    distribution_channel_cancellation_distribution(df)
    room_type_cancellation(df)
    correlation(df)
    

Dataset Overview:
<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 33 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal     